# AgentCore on AWS: Runtime + Gateway + MCP + A2A, really deployed

This notebook is the **AWS version** of `agentcore_runtime_gateway_a2a.ipynb`. The same three pieces (order MCP server,
shipment A2A agent, supervisor) now run **on AWS**, deployed with Terraform from `infra/agentcore-demo/`.

Anything that differs from the local notebook is marked with a callout like this:

> 🔶 **AWS CHANGE** — what is different compared with the local setup, and why.

## The picture

```
 you (boto3, IAM user)
      |  InvokeAgentRuntime (SigV4)
      v
 SUPERVISOR  (AgentCore Runtime, protocol HTTP, code zip from S3)
      |  1. route(message)
      |
      |-- order question ----SigV4----> AgentCore GATEWAY (inbound auth: IAM)
      |                                      |  gets an OAuth token for the target (AgentCore Identity + Cognito)
      |                                      v
      |                                 ORDER MCP SERVER (AgentCore Runtime, protocol MCP, inbound auth: JWT)
      |
      '-- shipping question --SigV4----> SHIPMENT AGENT (AgentCore Runtime, protocol A2A, inbound auth: IAM)
```

## Local vs AWS at a glance

| | Local notebook | This notebook (AWS) |
|---|---|---|
| Where code runs | `subprocess` on your laptop | **AgentCore Runtime** (arm64 microVMs), code zip in **S3** (no Docker needed) |
| Ports | 8001 / 9000 / 8090 | fixed by protocol: **8000** (MCP), **9000** (A2A), **8080** (HTTP), bound to `0.0.0.0` |
| Order agent -> tools | direct URL to the MCP server | via **AgentCore Gateway** (one URL, IAM auth, tool names get a target prefix) |
| Gateway -> MCP server | not present | **OAuth client-credentials token** from Cognito, minted by AgentCore Identity |
| Calls between services | plain HTTP | **SigV4-signed** HTTPS (IAM) |
| Calling the supervisor | `POST http://127.0.0.1:8090/invocations` | `bedrock-agentcore:InvokeAgentRuntime` API |
| Brain | keyword rules | **Groq `openai/gpt-oss-20b`** (routing + MCP tool choice), keyword rules as fallback |
| Infrastructure | none | Terraform: S3, IAM roles, Cognito, 3 Runtimes, Gateway + target |

## Cost

All of it is pay-per-use: Runtime bills active CPU/memory seconds (idle costs nothing), Gateway bills per call, Cognito and S3 are
pennies at demo volume. The last section destroys everything. **The cleanup is not optional hygiene: run it when you are done.**

## Steps

0. Setup and prerequisites  1. Write the service code  2. Local smoke test (free)  3. Build the deployment zip
4. Terraform apply  5. Verify every layer on AWS  6. Debugging with logs  7. Destroy

# Part 0. Setup

## 0.1 Imports, logging and settings

> 🔶 **AWS CHANGE** — new settings: the AWS region, the AWS CLI profile (`None` = default credentials) and the location of the
> Terraform folder. The Neon connection string and the Groq API key are read from the repo `.env` (only those two entries; they are only ever put in environment variables, never printed). The LLM model name comes from `config/base.yaml`. Everything else is the same helper code as before.

In [ ]:
import json
import logging
import os
import shutil
import socket
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import httpx
import yaml
from dotenv import dotenv_values

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logger = logging.getLogger("notebook")
for noisy in ("httpx", "mcp", "a2a", "httpcore", "botocore", "urllib3"):   # hide per-request chatter
    logging.getLogger(noisy).setLevel(logging.WARNING)

REGION = "us-east-1"                  # AgentCore Runtime / Gateway region
AWS_PROFILE = None                    # e.g. "default"; None = standard credential chain
PYTHON_VERSION = "3.12"               # must match var.python_runtime (PYTHON_3_12) in Terraform

DEMO_DIR = Path("agentcore_aws_demo")                    # everything this notebook generates
APP_DIR = DEMO_DIR / "app"                               # the service files (become the zip)
BUILD_DIR = DEMO_DIR / "build"                           # dependency install + zip
TF_DIR = Path("../infra/agentcore-demo").resolve()       # Terraform for this stack
APP_DIR.mkdir(parents=True, exist_ok=True)

# Take ONLY the Neon connection string from .env. Loading the whole file would also import its S3 keys and
# AWS_ENDPOINT_URL_S3 (meant for Neon storage) and silently override your real AWS credentials.
neon_dsn = dotenv_values(Path("../.env")).get("NEON_POSTGRES_CONNECTION_STRING") or os.environ.get("NEON_POSTGRES_CONNECTION_STRING")
if not neon_dsn:
    raise RuntimeError("NEON_POSTGRES_CONNECTION_STRING missing: set it in the repo .env")
os.environ["NEON_POSTGRES_CONNECTION_STRING"] = neon_dsn          # inherited by the local smoke-test services (never logged)

# LLM: model name comes from the project config (role "small"); the key is the only other thing taken from .env.
GROQ_MODEL = yaml.safe_load(Path("../config/base.yaml").read_text())["llm"]["small"]
groq_key = dotenv_values(Path("../.env")).get("GROQ_API_KEY") or os.environ.get("GROQ_API_KEY") or ""
os.environ["GROQ_API_KEY"] = groq_key                             # inherited by the local smoke-test services (never logged)
os.environ["GROQ_MODEL"] = GROQ_MODEL

if AWS_PROFILE:
    os.environ["AWS_PROFILE"] = AWS_PROFILE
os.environ["AWS_DEFAULT_REGION"] = REGION


def show(title: str, value: object = "") -> None:
    """Log a titled result so notebook output stays readable.

    Args:
        title: Short label for the line.
        value: Anything printable that belongs to the label.
    """
    logger.info("%s %s", title, value)

## 0.2 Check the prerequisites

> 🔶 **AWS CHANGE** — we now need three tools and working AWS credentials: `terraform` (infrastructure), `uv` (to download
> *Linux arm64* wheels, because Runtime runs on arm64 and your laptop does not), and credentials that may create IAM roles,
> Runtime, Gateway, Cognito and S3 resources.

In [ ]:
import boto3

for tool in ("terraform", "uv"):
    show(f"{tool:10}", shutil.which(tool) or "NOT FOUND -> install it and restart the kernel")

identity = boto3.client("sts", region_name=REGION).get_caller_identity()
ACCOUNT_ID = identity["Account"]
show("LLM model :", f"{GROQ_MODEL} via Groq (key {'found' if groq_key else 'MISSING -> keyword rules'})")
show("AWS account:", ACCOUNT_ID)
show("Caller     :", identity["Arn"])

# Part 1. The service code (three small files + one shared helper)

Each file is written from this notebook with `%%writefile`. They are the local files from the previous notebook with a few
**environment switches** added, so the *same code* runs on your laptop (defaults) and on AWS (Terraform sets the variables).

## 1.1 Shared helper: SigV4 signing

> 🔶 **AWS CHANGE (new file)** — locally nothing was authenticated. On AWS, the Gateway and the shipment Runtime use **IAM
> authentication**: every HTTP request must be *signed* with the caller's AWS credentials (Signature Version 4).
> `SigV4Auth` is a small `httpx.Auth` that does this with `botocore`. Inside a Runtime, the credentials are those of the
> Runtime's **execution role**; in this notebook they are your own.
>
> Service name for signing is `bedrock-agentcore` for both Gateway and Runtime.

In [ ]:
%%writefile agentcore_aws_demo/app/aws_common.py
"""Helpers shared by the supervisor and the notebook: env flags and SigV4 request signing."""

import os
import uuid

import httpx
from botocore.auth import SigV4Auth as BotocoreSigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.session import Session

SESSION_HEADER = "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id"  # Runtime wants a session id of 33+ characters


def env_flag(name: str) -> bool:
    """Read a boolean switch from the environment.

    Args:
        name: Environment variable name.

    Returns:
        bool: True if the variable is set to true/1/yes.
    """
    return os.environ.get(name, "").strip().lower() in ("1", "true", "yes")


def new_session_id() -> str:
    """Create a Runtime session id (must be at least 33 characters).

    Returns:
        str: A random 36-character id.
    """
    return str(uuid.uuid4())


class SigV4Auth(httpx.Auth):
    """httpx authentication that signs each request with the current AWS credentials (SigV4)."""

    requires_request_body = True  # the exact body bytes are part of the signature

    def __init__(self, service: str = "bedrock-agentcore", region: str | None = None) -> None:
        """Remember what to sign for.

        Args:
            service: AWS signing name. Gateway and Runtime both use `bedrock-agentcore`.
            region: AWS region; defaults to AWS_REGION / AWS_DEFAULT_REGION / us-east-1.
        """
        self._service = service
        self._region = region or os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-east-1"
        self._session = Session()  # resolves credentials: env vars, profile, or the Runtime's role

    def auth_flow(self, request: httpx.Request):
        """Add the SigV4 headers to the request before it is sent.

        Args:
            request: The outgoing httpx request.

        Yields:
            httpx.Request: The same request, now carrying Authorization / X-Amz-* headers.
        """
        credentials = self._session.get_credentials().get_frozen_credentials()
        # Only content-type is signed besides host/x-amz-*: proxies may rewrite other headers.
        to_sign = {k: v for k, v in request.headers.items() if k.lower() == "content-type"}
        aws_request = AWSRequest(method=request.method, url=str(request.url), data=request.content, headers=to_sign)
        BotocoreSigV4Auth(credentials, self._service, self._region).add_auth(aws_request)
        for name in ("Authorization", "X-Amz-Date", "X-Amz-Security-Token"):
            if name in aws_request.headers:
                request.headers[name] = aws_request.headers[name]
        yield request

## 1.1b The LLM helper (Groq `openai/gpt-oss-20b`)

> 🔶 **NEW vs the previous notebooks** — until now the "brain" was keyword rules. Now a real model decides, using Groq's
> OpenAI-compatible API through plain `httpx` (no SDK, so there is nothing extra to package).
>
> - **Model**: the project's *small* model, read from `config/base.yaml` in Part 0 and passed to the services as `GROQ_MODEL`.
> - **Where the LLM is used**: (1) the supervisor classifies the message into an agent (1 call, JSON output); (2) the order agent
>   receives the **MCP tool schemas** and picks a tool (1 call), then words the final answer (1 call). The shipment agent stays
>   deterministic on purpose: its job is to show A2A streaming, not to burn tokens.
> - **Free tier**: Groq is rate limited, so calls are few, `temperature=0`, `reasoning_effort="low"`. A full run is about 5 calls.
> - **Safe fallback**: no key or a failed call logs a warning and uses the keyword rules, so the notebook never depends on Groq being up.
> - 🔶 **AWS CHANGE**: the API key becomes a *sensitive Terraform variable* -> environment variable of the supervisor Runtime
>   (same demo-level handling as the Neon string; Secrets Manager in production). Runtime's `PUBLIC` network lets it reach `api.groq.com`.
> - Logging only records model, token counts and latency, never prompts or replies (they can contain customer data).

#### More about whats tools in this?

`tools` lets the model call functions you define instead of only answering in text. It's the standard OpenAI-compatible tool-calling parameter, which Groq uses too.

1. You pass a list of function specs. Each has a name, a description, and a JSON-schema for its arguments.
2. The model reads the user's message and decides whether it needs one of those functions.
3. If it does, the response has no text answer. It has `tool_calls` instead, naming the function and the arguments it chose.
4. Your code runs the function (an MCP tool, a database query, or an A2A call to another agent in this project). It then sends the result back as a `tool` message.
5. The model uses that result to write the final answer.

The model never runs anything itself. It only says which function to call and with what arguments, and your code does the call.

If `tools` is `None`, the model can only reply with text. That is why the docstring says "may answer with `tool_calls` *instead of* text". Callers have to check for both outcomes.

In this notebook, the AgentCore Gateway exposes tools such as product search over MCP. The agent turns those into `tools` specs so the LLM can pick them.

These are illustrative OpenAI/Groq-format payloads, not live output. I didn't call the API, to save the free-tier quota. The shapes are the standard chat completions format.

**Text answer** (the model needs no tool):

```json
{
  "choices": [{
    "finish_reason": "stop",
    "message": {
      "role": "assistant",
      "content": "Hi! I can help you find clothes, check orders, or get style tips.",
      "tool_calls": null
    }
  }]
}
```

**Tool call** (user asked "show me black jackets"):

```json
{
  "choices": [{
    "finish_reason": "tool_calls",
    "message": {
      " ": "assistant",
      "content": null,
      "tool_calls": [{
        "id": "call_abc123",
        "type": "function",
        "function": {
          "name": "search_products",
          "arguments": "{\"query\": \"black jacket\", \"limit\": 5}"
        }
      }]
    }
  }]
}
```

After you run the tool, you append the assistant message and a result message, then call the model again:

```json
{"role": "tool", "tool_call_id": "call_abc123", "content": "[{\"id\": \"prod003\", \"name\": \"Black biker jacket\"}]"}
```

The second call returns a normal text answer built from that result.

They're different objects. The first two are what the API sends back to you. The `tool` one is what you send to the API.

**Response (API → you)** is the whole HTTP response, so it has an envelope:
```
choices[] → each choice has finish_reason + message
```
`finish_reason` and `choices` describe how the generation went. They aren't part of the conversation. The actual chat turn is only the inner `message` object: `{"role": "assistant", "content": ..., "tool_calls": ...}`.

**Request (you → API)** has no envelope. You send a flat list of messages:
```python
messages = [
    {"role": "user", "content": "show me black jackets"},
    {"role": "assistant", "content": None, "tool_calls": [...]},   # the inner "message" from the response
    {"role": "tool", "tool_call_id": "call_abc123", "content": "..."},
]
client.chat.completions.create(model=..., messages=messages, tools=tools)
```

So the like-for-like comparison is the `message` object inside the response and the `tool` message:

| | assistant message (from response) | tool message (you build) |
|---|---|---|
| `role` | `"assistant"` | `"tool"` |
| `content` | text or `null` | the tool's result, as a string |
| `tool_calls` | present when calling a tool | not used |
| `tool_call_id` | not used | required; matches the `id` in the assistant's `tool_calls` |

`tool_call_id` links a result to the call that produced it. That matters when the model makes several calls at once.

The escaped quotes in `"content": "[{\"id\": ...}]"` are there because `content` must be a string. The tool's result is JSON, so it gets serialized with `json.dumps(result)` before it goes in. `arguments` in the assistant's tool call is a JSON string for the same reason.

In the notebook, the line you selected (`role`) is the key present in all three. Every message in the `messages` list needs it so the model knows who said what.

**NewQ: what is 'OpenAI-compatible tool-calling parameter'  and 'OpenAI-compatible API' here?**

"OpenAI-compatible" means the API copies the request and response format that OpenAI's Chat Completions API defined. It doesn't mean the call goes to OpenAI.

**OpenAI-compatible API.** OpenAI's Chat Completions endpoint (`POST /v1/chat/completions`) became the de facto standard for LLM APIs. It takes `model`, `messages` (a list of `{role, content}`), and optional `tools`, and it returns `choices[0].message`. Many other providers expose the same endpoint shape, including Groq, Together, vLLM, Ollama and OpenRouter. In this project, Groq serves `openai/gpt-oss-20b` through such an endpoint, so you can use the OpenAI client library and only change two things:

```python
from openai import OpenAI
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)
```

The URL and key point at Groq. The code and the message formats are the same as for OpenAI.

**OpenAI-compatible tool-calling parameter.** This is the `tools` parameter in that same format. Each tool is declared like this:

```json
{
  "type": "function",
  "function": {
    "name": "search_products",
    "description": "Search the H&M catalog",
    "parameters": { "type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"] }
  }
}
```

The response side is `message.tool_calls` and `finish_reason: "tool_calls"`, plus the `role: "tool"` message you send back. This is the exact format from my earlier examples.

Because of this shared format, the notebook's function can talk to any provider that follows it. You can swap models or providers by changing `base_url` and `model`, which fits the project's rule of setting model names per role in `config/`. Anthropic's native API uses a different format (`tool_use` blocks), which is why the compatibility matters here.

#### code

In [ ]:
%%writefile agentcore_aws_demo/app/llm.py
"""Tiny Groq chat helper (OpenAI-compatible API over httpx): one function, no SDK, nothing logged but aggregates."""

import logging
import os
import time

import httpx

logger = logging.getLogger(__name__)

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "openai/gpt-oss-20b"   # the project's "small" model (config/base.yaml); Terraform passes GROQ_MODEL on AWS


def groq_enabled() -> bool:
    """Tell whether an LLM key is configured; callers fall back to keyword rules when it is not.

    Returns:
        bool: True if GROQ_API_KEY is set and non-empty.
    """
    return bool(os.environ.get("GROQ_API_KEY", "").strip())


async def chat(messages: list[dict], tools: list[dict] | None = None, json_mode: bool = False) -> dict:
    """Send one chat request to Groq and return the assistant message.

    Free-tier friendly: temperature 0 and low reasoning effort keep the small model fast and cheap.
    Only model name, token counts and latency are logged, never the prompt or the reply (they can hold customer data).

    Args:
        messages: OpenAI-style messages (`{"role": ..., "content": ...}`).
        tools: Optional function specs; if given, the model may answer with `tool_calls` instead of text.
        json_mode: Force the reply to be a JSON object (the prompt must mention the word JSON).

    Returns:
        dict: The assistant message: `content` (text) and, when the model chose a tool, `tool_calls`.

    Raises:
        httpx.HTTPError: On network errors or a non-2xx status (for example 429 when the free-tier limit is hit).
    """
    model = os.environ.get("GROQ_MODEL") or DEFAULT_MODEL
    payload: dict = {"model": model, "messages": messages, "temperature": 0, "reasoning_effort": "low"}
    if tools:
        payload.update(tools=tools, tool_choice="auto")
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    start = time.perf_counter()
    async with httpx.AsyncClient(timeout=30) as http:
        response = await http.post(GROQ_URL, json=payload, headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"})
        response.raise_for_status()
    data = response.json()
    usage = data.get("usage", {})
    logger.info("llm model=%s prompt_tokens=%s completion_tokens=%s duration_s=%.2f",
                model, usage.get("prompt_tokens"), usage.get("completion_tokens"), time.perf_counter() - start)
    return data["choices"][0]["message"]

## 1.2 The order MCP server

Same three tools as in the local notebook, but the fake dictionary is gone: they now run **read-only SQL on your real Neon
database** (`order_management.orders` / `order_items`, the same tables the project's MCP server uses).

> 🔶 **AWS CHANGE** — the *start-up* lines, plus the data source:
>
> - **Data**: queries Neon through `psycopg`. The connection string comes from `NEON_POSTGRES_CONNECTION_STRING`; locally the notebook passes it from your `.env`, on AWS Terraform sets it on the Runtime. Queries are parameterised (`%s`) and filter by `customer_id`, so one customer cannot read another's order.
> - **Secret handling (demo level)**: the connection string becomes a Runtime environment variable and sits in the Terraform state file. That is fine for a personal demo; in production keep it in **AWS Secrets Manager** and read it at start-up.
> - Neon is reached over the public internet, which Runtime's `PUBLIC` network mode allows.
>
> | | Local | AWS Runtime (protocol `MCP`) |
> |---|---|---|
> | host / port | `127.0.0.1:8001` | **`0.0.0.0:8000`** (Runtime requires it; Terraform sets `HOST`, `PORT`) |
> | `stateless_http` | no | **yes**: Runtime may route each request to a fresh microVM, so the server must not keep session state |
> | path | `/mcp` | `/mcp` (Runtime proxies `.../invocations` to it) |
> | who may call it | anyone | only callers with a valid **Cognito JWT** (`authorizer_configuration` in Terraform; the Gateway sends it) |
>
> The code does not check tokens itself: **Runtime validates the JWT before the request reaches your code**.

In [ ]:
%%writefile agentcore_aws_demo/app/order_mcp_server.py
"""Order MCP server: three read-only tools over the Neon Postgres `order_management` schema."""

import logging
import os
import sys

import psycopg
from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[order-mcp] %(message)s")
logger = logging.getLogger(__name__)

# Defaults are the local setup; on AgentCore Runtime Terraform sets HOST=0.0.0.0, PORT=8000, STATELESS_HTTP=true.
mcp = FastMCP(
    "order-management",
    host=os.environ.get("HOST", "127.0.0.1"),
    port=int(os.environ.get("PORT", "8001")),
    stateless_http=os.environ.get("STATELESS_HTTP", "").lower() == "true",
)


def fetch_all(sql: str, params: tuple) -> list[tuple]:
    """Run one read-only query on Neon and return all rows.

    A new connection per call keeps the server stateless (Runtime may start a fresh microVM for any request).

    Args:
        sql: SQL text with %s placeholders.
        params: Values for the placeholders (never string-formatted into the SQL).

    Returns:
        list[tuple]: The result rows.

    Raises:
        KeyError: If NEON_POSTGRES_CONNECTION_STRING is not set.
    """
    with psycopg.connect(os.environ["NEON_POSTGRES_CONNECTION_STRING"], connect_timeout=15) as conn:
        return conn.execute(sql, params).fetchall()


@mcp.tool()
def list_orders(customer_id: str) -> str:
    """List a customer's most recent orders (id, date, channel, number of items), newest first.

    Args:
        customer_id: Customer id such as cust001.
    """
    rows = fetch_all(
        """
        SELECT o.order_id, o.order_date, o.sales_channel, COUNT(i.line_no)
        FROM order_management.orders o JOIN order_management.order_items i USING (order_id)
        WHERE o.customer_id = %s
        GROUP BY o.order_id, o.order_date, o.sales_channel
        ORDER BY o.order_date DESC, o.order_id DESC LIMIT 5
        """,
        (customer_id,),
    )
    logger.info("tool=list_orders orders=%d", len(rows))
    if not rows:
        return "No orders found for this customer."
    return "\n".join(f"{oid}: {date} ({channel}), {n} item(s)" for oid, date, channel, n in rows)


@mcp.tool()
def get_order_details(customer_id: str, order_id: str) -> str:
    """Get the date, channel and items of one order. Only finds orders that belong to the given customer.

    Args:
        customer_id: Customer id such as cust001.
        order_id: Order id such as ORD-1007.
    """
    rows = fetch_all(
        """
        SELECT o.order_date, o.sales_channel, i.product_name
        FROM order_management.orders o JOIN order_management.order_items i USING (order_id)
        WHERE o.customer_id = %s AND o.order_id = %s
        ORDER BY i.line_no
        """,
        (customer_id, order_id.upper()),
    )
    logger.info("tool=get_order_details items=%d", len(rows))
    if not rows:
        return "No order with that id was found for this customer."
    order_date, channel, _ = rows[0]
    return f"Order {order_id.upper()} placed on {order_date} ({channel}), {len(rows)} item(s): {', '.join(r[2] for r in rows)}."


@mcp.tool()
def get_last_bought_product(customer_id: str) -> str:
    """Get the name of the last product a customer bought.

    Args:
        customer_id: Customer id such as cust001.
    """
    rows = fetch_all(
        """
        SELECT i.product_name
        FROM order_management.orders o JOIN order_management.order_items i USING (order_id)
        WHERE o.customer_id = %s
        ORDER BY o.order_date DESC, o.order_id DESC, i.line_no DESC LIMIT 1
        """,
        (customer_id,),
    )
    return rows[0][0] if rows else "No purchases found for this customer."


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

## 1.3 The shipment tracking A2A agent

Same Agent Card, same task flow (`submitted` -> `working` progress lines -> artifact -> `completed`).

> 🔶 **AWS CHANGE**
>
> | | Local | AWS Runtime (protocol `A2A`) |
> |---|---|---|
> | host / port | `127.0.0.1:9000` | **`0.0.0.0:9000`** |
> | `GET /ping` | not needed | **required** health check, so we added the route |
> | public URL | `http://127.0.0.1:9000` | `https://bedrock-agentcore.<region>.amazonaws.com/runtimes/<encoded-arn>/invocations/` |
> | auth | none | **IAM (SigV4)** — callers need `bedrock-agentcore:InvokeAgentRuntime` |
>
> **Gotcha 1 (found while deploying this):** Runtime's A2A proxy **does not forward the `A2A-Version` header**. The SDK reads a
> missing header as protocol 0.3 and answers `VersionNotSupportedError`. The small `DefaultA2AVersion` middleware below treats a missing
> header as `1.0`. Locally the header arrives untouched, so it changes nothing there.
>
> **Gotcha 2:** the Agent Card contains the agent's own URL, but the agent cannot know its Runtime ARN when it starts.
> So the *client* (our supervisor) overrides the URL from the card with the real Runtime URL (see 1.4).

In [ ]:
%%writefile agentcore_aws_demo/app/shipment_a2a_server.py
"""Shipment tracking agent exposed over A2A (simulated carrier lookup)."""

import asyncio
import hashlib
import logging
import os
import re

import uvicorn
from a2a.helpers.proto_helpers import new_task_from_user_message, new_text_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events.event_queue_v2 import EventQueue
from a2a.server.request_handlers import DefaultRequestHandlerV2
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types.a2a_pb2 import AgentCapabilities, AgentCard, AgentInterface, AgentSkill, TaskState
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH, PROTOCOL_VERSION_CURRENT, TransportProtocol
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route

logging.basicConfig(level=logging.INFO, format="[shipment-a2a] %(message)s")
logger = logging.getLogger(__name__)

# Defaults are the local setup; on AgentCore Runtime Terraform sets HOST=0.0.0.0, PORT=9000.
HOST = os.environ.get("HOST", "127.0.0.1")
PORT = int(os.environ.get("PORT", "9000"))
ORDER_ID_PATTERN = re.compile(r"\bORD-\d+\b", re.IGNORECASE)
CARRIERS = [
    ("DHL Express", "Hamburg sorting hub", "In transit, expected delivery in 2 days"),
    ("PostNord", "Stockholm distribution centre", "Out for delivery, expected today"),
    ("UPS", "Rotterdam depot", "Held at the depot, delivery attempt tomorrow"),
]


def build_card() -> AgentCard:
    """Build the Agent Card: the public self-description clients read before talking to us.

    Returns:
        AgentCard: Card with one skill; `streaming=True` tells clients they can watch progress live.
    """
    return AgentCard(
        name="shipment_tracking",
        description="Checks the carrier for the shipping status of an order. The lookup is slow, so it streams progress.",
        version="1.0.0",
        supported_interfaces=[
            AgentInterface(
                url=os.environ.get("AGENT_URL", f"http://127.0.0.1:{PORT}"),  # the client overrides this on AWS
                protocol_binding=TransportProtocol.JSONRPC,
                protocol_version=PROTOCOL_VERSION_CURRENT,
            )
        ],
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[
            AgentSkill(
                id="shipment-status", name="Shipment status lookup",
                description="Carrier, last scan and delivery outlook for an order id.",
                tags=["shipping"], examples=["Where is my order ORD-1007?"],
            )
        ],
    )


class ShipmentExecutor(AgentExecutor):
    """The agent's work: turn one incoming message into a streamed task."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        """Run one task: announce it, report progress, publish the answer, complete.

        Args:
            context: Incoming message plus task/context ids and request metadata.
            event_queue: Where task events are published (the SDK streams them to the caller).
        """
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        if context.current_task is None:
            await event_queue.enqueue_event(new_task_from_user_message(context.message))  # state: submitted
        await updater.start_work()                                                         # state: working

        match = ORDER_ID_PATTERN.search(context.get_user_input())
        if not match:
            await updater.add_artifact([new_text_part("Which order should I track? Send an id like ORD-1007.")],
                                       artifact_id="reply", name="reply", last_chunk=True)
            await updater.complete()
            return

        order_id = match.group(0).upper()
        logger.info("task=%s order=%s status=start", context.task_id, order_id)
        for step in ["Contacting the carrier", "Waiting for the carrier system", "Reading the latest scan events"]:
            await updater.update_status(TaskState.TASK_STATE_WORKING, new_text_message(step))  # progress line
            await asyncio.sleep(1)                                                             # the "slow" part

        digest = int(hashlib.sha256(order_id.encode()).hexdigest()[:8], 16)
        carrier, location, outlook = CARRIERS[digest % len(CARRIERS)]
        reply = f"Order {order_id} is with {carrier}. Last scan: {location}. {outlook}."
        await updater.add_artifact([new_text_part(reply)], artifact_id="reply", name="reply", last_chunk=True)
        await updater.complete()                                                               # state: completed
        logger.info("task=%s status=complete", context.task_id)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        """Mark the task cancelled (our runs are short, so there is nothing to stop).

        Args:
            context: Request context holding the task to cancel.
            event_queue: Queue the cancellation event is published to.
        """
        await TaskUpdater(event_queue, context.task_id, context.context_id).cancel()


async def ping(_request) -> JSONResponse:
    """Health check required by AgentCore Runtime (GET /ping).

    Args:
        _request: The incoming request (unused).

    Returns:
        JSONResponse: Status "Healthy".
    """
    return JSONResponse({"status": "Healthy"})


class DefaultA2AVersion:
    """ASGI middleware: treat requests without an `A2A-Version` header as protocol 1.0.

    AgentCore Runtime's A2A proxy does not forward that header, and the SDK reads a missing header as the old
    0.3 protocol and rejects the call. Locally the client's header arrives untouched, so this changes nothing there.
    """

    def __init__(self, inner) -> None:
        """Wrap the real app.

        Args:
            inner: The ASGI application to protect.
        """
        self._inner = inner

    async def __call__(self, scope, receive, send) -> None:
        """Add the version header when it is missing, then hand the request on.

        Args:
            scope: ASGI connection scope.
            receive: ASGI receive channel.
            send: ASGI send channel.
        """
        if scope["type"] == "http" and not any(name == b"a2a-version" for name, _ in scope["headers"]):
            scope["headers"] = [*scope["headers"], (b"a2a-version", b"1.0")]
        await self._inner(scope, receive, send)


card = build_card()
handler = DefaultRequestHandlerV2(agent_executor=ShipmentExecutor(), task_store=InMemoryTaskStore(), agent_card=card)
starlette_app = Starlette(routes=[
    Route("/ping", ping),                                                  # GET /ping  (Runtime health check)
    *create_agent_card_routes(card, card_url=AGENT_CARD_WELL_KNOWN_PATH),  # GET /.well-known/agent-card.json
    *create_jsonrpc_routes(handler, rpc_url="/"),                          # POST /  (JSON-RPC messages)
])
app = DefaultA2AVersion(starlette_app)

if __name__ == "__main__":
    uvicorn.run(app, host=HOST, port=PORT, log_level="warning")

## 1.4 The supervisor (with the order agent inside)

Same flow as before (route -> order agent via MCP, or shipment agent via A2A), but the decisions are now made by the LLM from 1.1b:
>
> - `route()` asks the model for `{"agent": ..., "reason": ...}`; invalid answers or errors fall back to `route_by_keywords()`.
> - `llm_order_agent()` turns the MCP tools into function specs, **hides `customer_id` from the model**, lets it pick a tool, then **overwrites `customer_id` with the system's value** before calling the tool. The model never chooses whose data is read.

> 🔶 **AWS CHANGE**
>
> 1. **URLs and signing come from environment variables** that Terraform sets: `ORDER_MCP_URL` (now the *Gateway* URL),
>    `SHIPMENT_A2A_URL` (the shipment *Runtime* URL), and `*_SIGV4=true` to sign the calls.
> 2. **Gateway prefixes tool names** with the target name: `list_orders` becomes `order-management___list_orders`
>    (three underscores). The agent finds a tool by its *suffix* (`find_tool`), so it works both locally and through the Gateway.
> 3. **A2A card URL override**: after fetching the card we replace its URL with `SHIPMENT_A2A_URL` (see 1.3).
> 4. **Session header** `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` is sent on Runtime calls (Runtime groups requests by session).
> 5. `app.run(host=..., port=...)` binds to **`0.0.0.0:8080`** on Runtime (the HTTP protocol port) instead of `127.0.0.1:8090`.
> 6. **Permissions** are not in the code: the Runtime's execution role (Terraform) allows `InvokeGateway` and `InvokeAgentRuntime`.

In [ ]:
%%writefile agentcore_aws_demo/app/supervisor_runtime.py
"""Supervisor as an AgentCore Runtime app: an LLM routes the message to the order agent (MCP) or shipment agent (A2A)."""

import json
import logging
import os
import re
import time

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.helpers.proto_helpers import get_artifact_text, get_message_text, new_text_part
from a2a.types.a2a_pb2 import Message, Role, SendMessageRequest, TaskState
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from aws_common import SESSION_HEADER, SigV4Auth, env_flag, new_session_id
from llm import chat, groq_enabled

logging.basicConfig(level=logging.INFO, format="[supervisor] %(message)s")
logger = logging.getLogger(__name__)

# Where the other pieces live. Local defaults: direct URLs, no signing. On AWS Terraform sets all four.
ORDER_MCP_URL = os.environ.get("ORDER_MCP_URL", "http://127.0.0.1:8001/mcp")   # AWS: the Gateway URL (ends in /mcp)
ORDER_MCP_SIGV4 = env_flag("ORDER_MCP_SIGV4")                                   # AWS: true (Gateway uses IAM auth)
SHIPMENT_A2A_URL = os.environ.get("SHIPMENT_A2A_URL", "http://127.0.0.1:9000")  # AWS: shipment Runtime URL
SHIPMENT_SIGV4 = env_flag("SHIPMENT_SIGV4")                                     # AWS: true (Runtime uses IAM auth)
ORDER_ID_PATTERN = re.compile(r"\bORD-\d+\b", re.IGNORECASE)
AGENTS = ("order_management", "shipment_tracking", "out_of_scope")

ROUTER_PROMPT = """You are the supervisor of the H&M customer support team. You never answer the customer; you only choose
which specialist handles the latest message. Reply with JSON only: {"agent": "<name>", "reason": "<one short sentence>"}.
Agents:
- order_management: the customer's own orders or purchases (what they bought, last purchase, order history, order contents).
- shipment_tracking: where an order or parcel is now, shipping or delivery status.
- out_of_scope: anything else."""

ORDER_AGENT_PROMPT = """You are the H&M order management agent. Answer the customer's question by calling exactly one of the
available tools. The customer is already identified by the system: never ask for or use a customer id.
If the customer names an order id (like ORD-1007) use it. Never invent order data.
After you receive the tool result, reply in 1-3 friendly sentences."""

app = BedrockAgentCoreApp()  # provides GET /ping and POST /invocations


def route_by_keywords(message: str) -> str:
    """Fallback router (no LLM): decide the agent from keywords.

    Args:
        message: The customer's message.

    Returns:
        str: `shipment_tracking`, `order_management` or `out_of_scope`.
    """
    text = message.lower()
    if any(word in text for word in ("where", "track", "deliver", "parcel", "shipping", "shipment")):
        return "shipment_tracking"
    if any(word in text for word in ("order", "bought", "buy", "purchase")):
        return "order_management"
    return "out_of_scope"


async def route(message: str) -> str:
    """Decide which agent should handle the message: ask the LLM, fall back to keyword rules if that is not possible.

    Args:
        message: The customer's message.

    Returns:
        str: One of `AGENTS`.
    """
    if groq_enabled():
        try:
            reply = await chat([{"role": "system", "content": ROUTER_PROMPT}, {"role": "user", "content": message}], json_mode=True)
            decision = json.loads(reply["content"])
            if decision.get("agent") in AGENTS:
                logger.info("route source=llm agent=%s", decision["agent"])
                return decision["agent"]
            logger.warning("route source=llm status=invalid_agent")
        except Exception:  # rate limit, network, bad JSON: a cheap rule is better than failing the customer's request
            logger.exception("route source=llm status=failed, using keyword rules")
    return route_by_keywords(message)


def find_tool(tool_names: list[str], wanted: str) -> str:
    """Find a tool by name, also when a Gateway added a target prefix (`order-management___list_orders`).

    Args:
        tool_names: Names returned by `list_tools()`.
        wanted: The plain tool name, e.g. `list_orders`.

    Returns:
        str: The real name to call.

    Raises:
        LookupError: If no tool matches.
    """
    for name in tool_names:
        if name == wanted or name.endswith(f"___{wanted}"):
            return name
    raise LookupError(f"tool {wanted!r} not offered; available: {tool_names}")


def tool_spec_without_customer(tool) -> dict:
    """Turn an MCP tool into an LLM function spec that hides `customer_id` from the model.

    The model must not choose whose data is read; the system supplies the customer id itself.

    Args:
        tool: An MCP tool as returned by `list_tools()`.

    Returns:
        dict: OpenAI-style function spec (name, description, JSON-schema parameters).
    """
    schema = json.loads(json.dumps(tool.inputSchema))  # deep copy
    schema.get("properties", {}).pop("customer_id", None)
    schema["required"] = [name for name in schema.get("required", []) if name != "customer_id"]
    return {"type": "function", "function": {"name": tool.name, "description": tool.description, "parameters": schema}}


async def llm_order_agent(session: ClientSession, tools: list, message: str, customer_id: str) -> str:
    """LLM order agent: the model picks an MCP tool from its schema, we call it, the model words the answer.

    Args:
        session: An initialised MCP session (to the MCP server or the Gateway).
        tools: Tools from `session.list_tools()`.
        message: The customer's message.
        customer_id: Customer id supplied by the system.

    Returns:
        str: The final reply text.
    """
    messages = [{"role": "system", "content": ORDER_AGENT_PROMPT}, {"role": "user", "content": message}]
    reply = await chat(messages, tools=[tool_spec_without_customer(t) for t in tools])
    calls = reply.get("tool_calls")
    if not calls:
        return reply.get("content") or "Sorry, I could not find that information."

    call = calls[0]
    args = json.loads(call["function"].get("arguments") or "{}")
    args["customer_id"] = customer_id   # always overwritten by the system, whatever the model wrote
    logger.info("order_agent source=llm tool=%s", call["function"]["name"])
    result = await session.call_tool(call["function"]["name"], args)
    tool_text = result.content[0].text

    messages += [
        {"role": "assistant", "content": reply.get("content") or "", "tool_calls": calls},
        {"role": "tool", "tool_call_id": call["id"], "content": tool_text},
    ]
    final = await chat(messages)
    return final.get("content") or tool_text


async def run_order_agent(message: str, customer_id: str) -> str:
    """Order agent: connect to the MCP endpoint (the Gateway on AWS) and answer with the LLM, or with rules as fallback.

    Args:
        message: The customer's message.
        customer_id: Customer id supplied by the system (never by the model).

    Returns:
        str: The reply text.
    """
    auth = SigV4Auth() if ORDER_MCP_SIGV4 else None
    async with streamablehttp_client(ORDER_MCP_URL, auth=auth) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            if groq_enabled():
                try:
                    return await llm_order_agent(session, tools, message, customer_id)
                except Exception:
                    logger.exception("order_agent source=llm status=failed, using keyword rules")
            names = [t.name for t in tools]
            order_id = ORDER_ID_PATTERN.search(message)
            if order_id:
                wanted, args = "get_order_details", {"order_id": order_id.group(0).upper()}
            elif "last" in message.lower():
                wanted, args = "get_last_bought_product", {}
            else:
                wanted, args = "list_orders", {}
            tool = find_tool(names, wanted)
            logger.info("order_agent source=rules tool=%s", tool)
            result = await session.call_tool(tool, {"customer_id": customer_id, **args})
            return result.content[0].text


async def run_shipment_agent(message: str, customer_id: str):
    """Call the shipment agent over A2A and yield its progress lines and answer pieces.

    Args:
        message: The customer's message.
        customer_id: Customer id, forwarded as request metadata.

    Yields:
        tuple[str, str]: `("progress", text)` or `("answer", text)`.
    """
    auth = SigV4Auth() if SHIPMENT_SIGV4 else None
    async with httpx.AsyncClient(timeout=60, auth=auth, headers={SESSION_HEADER: new_session_id()}) as http:
        card = await A2ACardResolver(http, SHIPMENT_A2A_URL).get_agent_card()
        card.supported_interfaces[0].url = SHIPMENT_A2A_URL  # the card cannot know its own Runtime URL
        client = ClientFactory(ClientConfig(streaming=card.capabilities.streaming, httpx_client=http)).create(card)
        request = SendMessageRequest(
            message=Message(message_id=f"msg-{time.time_ns()}", role=Role.ROLE_USER, parts=[new_text_part(message)]),
            metadata={"customer_id": customer_id},
        )
        async for response in client.send_message(request):
            if response.HasField("status_update"):
                status = response.status_update.status
                if status.state == TaskState.TASK_STATE_WORKING and status.message.parts:
                    yield "progress", get_message_text(status.message)
            elif response.HasField("artifact_update"):
                yield "answer", get_artifact_text(response.artifact_update.artifact, delimiter="")
        await client.close()


@app.entrypoint
async def invoke(payload: dict, context):
    """Runtime entrypoint: handle one customer message and stream events.

    Args:
        payload: Request body, `{"message": "...", "customer_id": "cust001"}`.
        context: Runtime request context; `context.session_id` is the conversation id.

    Yields:
        dict: Events: `agent_selected`, `progress`, `reply`, then `done`.
    """
    message = str(payload.get("message", ""))
    customer_id = str(payload.get("customer_id", "cust001"))
    agent = await route(message)
    logger.info("session=%s agent=%s", getattr(context, "session_id", ""), agent)
    yield {"type": "agent_selected", "agent": agent}

    try:
        if agent == "order_management":
            yield {"type": "reply", "agent": agent, "text": await run_order_agent(message, customer_id)}
        elif agent == "shipment_tracking":
            reply = ""
            async for kind, text in run_shipment_agent(message, customer_id):
                if kind == "progress":
                    yield {"type": "progress", "agent": agent, "text": text}
                else:
                    reply += text
            yield {"type": "reply", "agent": agent, "text": reply}
        else:
            yield {"type": "reply", "agent": agent, "text": "I can help with orders and parcel tracking."}
    except Exception as exc:  # show the cause while we are learning; a real service would return a generic message
        logger.exception("agent=%s failed", agent)
        yield {"type": "error", "agent": agent, "text": f"{type(exc).__name__}: {exc}"}
    yield {"type": "done"}


if __name__ == "__main__":
    # Local defaults 127.0.0.1:8090 (8080 is often taken); on Runtime Terraform sets HOST=0.0.0.0, PORT=8080.
    app.run(host=os.environ.get("HOST", "127.0.0.1"), port=int(os.environ.get("PORT", "8090")))

# Part 2. Local smoke test (free)

This uses the real LLM too (about 4 small Groq calls). Before spending anything on AWS we prove the files still work **locally with their defaults** (no env vars set = no signing,
direct URLs). If this passes, any later failure is about AWS wiring, not the code.

> 🔶 **AWS CHANGE** — none to the code path; this part exists so deployment problems are not mixed with code problems.
> (The same discipline as the project rule: finish local testing before deploying.)

In [ ]:
processes: dict[str, subprocess.Popen] = {}


def port_open(port: int) -> bool:
    """Check whether something is listening on a local port.

    Args:
        port: TCP port on 127.0.0.1.

    Returns:
        bool: True if a connection succeeds.
    """
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) == 0


def start_service(name: str, script: str, port: int) -> None:
    """Start one of the service files as a background process and wait for its port.

    Args:
        name: Label for the log file and the registry.
        script: File inside `agentcore_aws_demo/app/`.
        port: Port the service listens on locally.

    Raises:
        RuntimeError: If the port is busy beforehand or does not open within 20 seconds.
    """
    if port_open(port):
        raise RuntimeError(f"Port {port} is already in use by another program.")
    log_path = DEMO_DIR / f"{name}.log"
    env = {k: v for k, v in os.environ.items() if k not in ("HOST", "PORT")}   # local defaults
    processes[name] = subprocess.Popen([sys.executable, "-u", script], cwd=APP_DIR.resolve(),
                                       stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=env)
    for _ in range(40):
        if port_open(port):
            show(f"started {name}", f"port {port}")
            return
        time.sleep(0.5)
    raise RuntimeError(f"{name} did not start:\n{log_path.read_text()[-1500:]}")


def stop_all() -> None:
    """Stop every background process started by `start_service`."""
    for name, process in list(processes.items()):
        if process.poll() is None:
            process.terminate()
            process.wait(timeout=10)
        processes.pop(name)


def invoke_local(message: str, customer_id: str = "cust001") -> None:
    """POST a message to the local supervisor and log each streamed event.

    Args:
        message: The customer's message.
        customer_id: Customer id sent in the body.
    """
    body = {"message": message, "customer_id": customer_id}
    headers = {"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": "local-session-0000000000000000000001"}
    with httpx.stream("POST", "http://127.0.0.1:8090/invocations", json=body, headers=headers, timeout=60) as response:
        for line in response.iter_lines():
            if line.startswith("data:"):
                event = json.loads(line.removeprefix("data:").strip())
                show(f"  {event.get('type'):15}", event.get("agent", "") + " " + event.get("text", ""))


try:
    start_service("order_mcp", "order_mcp_server.py", 8001)
    start_service("shipment_a2a", "shipment_a2a_server.py", 9000)
    start_service("supervisor", "supervisor_runtime.py", 8090)
    for question in ["What was my last purchase?", "Where is my order ORD-1007?"]:
        show(f"Q: {question}")
        invoke_local(question)
finally:
    stop_all()
    show("local services stopped")

# Part 3. Build the deployment package

> 🔶 **AWS CHANGE (whole part is new)** — locally Python imported libraries from your `.venv`. AgentCore Runtime has no
> `.venv`: you hand it a **zip with your code and all dependencies**, and it runs the `entry_point` file.
>
> Two details that trip people up:
>
> - Runtime runs **Linux on arm64**. Your laptop is Windows/x86, so we ask `uv` for wheels of the *target* platform
>   (`--python-platform aarch64-manylinux_2_28 --python-version 3.12 --only-binary=:all:`). No Docker needed.
> - File permissions inside the zip must be readable by Runtime (644 files, 755 dirs). Windows zips do not carry permissions,
>   so we set them explicitly while writing the zip.
>
> One zip serves all three Runtimes; each Runtime just names a different `entry_point` file (Terraform). The limit is 250 MB zipped.

In [ ]:
# Pinned below the next major version: an unpinned "mcp" resolved to 2.x, where FastMCP was renamed, and the server crashed on import.
REQUIREMENTS = ["mcp>=1.30,<2", "a2a-sdk[http-server]>=1.2.2,<2", "bedrock-agentcore>=1.25,<2", "httpx", "botocore", "psycopg[binary]>=3.3,<4"]
ZIP_PATH = BUILD_DIR / "app.zip"


def build_zip() -> Path:
    """Install linux/arm64 dependencies next to the service files and write them into one zip.

    Returns:
        Path: The zip file Terraform uploads to S3.

    Raises:
        subprocess.CalledProcessError: If the dependency install fails.
    """
    package_dir = BUILD_DIR / "package"
    shutil.rmtree(BUILD_DIR, ignore_errors=True)
    package_dir.mkdir(parents=True)

    subprocess.run(
        ["uv", "pip", "install", "--quiet", "--target", str(package_dir),
         "--python-platform", "aarch64-manylinux_2_28", "--python-version", PYTHON_VERSION,
         "--only-binary", ":all:", *REQUIREMENTS],
        check=True,
    )
    for source in APP_DIR.glob("*.py"):          # the service files go to the zip root (entry points)
        shutil.copy(source, package_dir / source.name)

    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(package_dir.rglob("*")):
            if "__pycache__" in path.parts:
                continue
            info = zipfile.ZipInfo(path.relative_to(package_dir).as_posix() + ("/" if path.is_dir() else ""))
            info.external_attr = (0o755 if path.is_dir() else 0o644) << 16     # POSIX permissions Runtime needs
            info.compress_type = zipfile.ZIP_DEFLATED
            archive.writestr(info, b"" if path.is_dir() else path.read_bytes())
    return ZIP_PATH


build_zip()
show("zip size (MB):", round(ZIP_PATH.stat().st_size / 1_000_000, 1))
with zipfile.ZipFile(ZIP_PATH) as z:
    show("entry points present:", [n for n in z.namelist() if n.endswith(".py") and "/" not in n])

# Part 4. Create the infrastructure with Terraform

> 🔶 **AWS CHANGE (whole part is new)** — this is the step that makes it real. The Terraform lives in
> [infra/agentcore-demo/](../infra/agentcore-demo/main.tf). Here is what each block creates and why:

| Resource (Terraform) | Purpose |
|---|---|
| `aws_s3_bucket` + `aws_s3_object` | Holds `app.zip`. Versioned, so a new zip makes Runtime redeploy. |
| `aws_iam_role` x3 (+ policies) | **Execution roles.** `runtime_basic` (MCP + shipment: logs, read zip), `runtime_supervisor` (adds `InvokeGateway`, `InvokeAgentRuntime`), `gateway` (fetch OAuth tokens). |
| `aws_cognito_user_pool` / `_domain` / `_resource_server` / `_client` | A tiny OAuth server. Its **client-credentials** token is how the Gateway proves itself to the MCP server. |
| `aws_bedrockagentcore_oauth2_credential_provider` | **AgentCore Identity** stores the Cognito client secret and mints tokens for the Gateway. |
| `var.groq_api_key` / `var.groq_model` | Sensitive key and model name, passed to the supervisor Runtime as environment variables for the LLM calls. |
| `var.neon_connection_string` (sensitive) | Passed to the MCP Runtime as an environment variable so its tools can query Neon. |
| `aws_bedrockagentcore_agent_runtime` x3 | The three Runtimes. Differences: `server_protocol` = `MCP` / `A2A` / `HTTP`; MCP has a **JWT authorizer**, the others default to **IAM**. |
| `aws_bedrockagentcore_gateway` | The front door; `authorizer_type = "AWS_IAM"` (callers sign with SigV4). |
| `aws_bedrockagentcore_gateway_target` | Registers the MCP Runtime URL as a target, with the OAuth credential provider as outbound auth. The Gateway lists its tools when this is created. |

Who trusts whom (the part that is hardest to see):

```
notebook user --IAM--> supervisor runtime        (needs InvokeAgentRuntime)
supervisor role --IAM--> gateway                 (policy: InvokeGateway)
supervisor role --IAM--> shipment runtime        (policy: InvokeAgentRuntime)
gateway --Cognito JWT--> MCP runtime             (JWT authorizer: allowed client = the gateway's Cognito client)
```

## 4.1 A helper to run Terraform and stream its output

In [ ]:
def terraform(*args: str) -> str:
    """Run a Terraform command in the stack folder, logging its output live.

    Args:
        *args: Terraform arguments, e.g. `"apply", "-auto-approve"`.

    Returns:
        str: The full output text.

    Raises:
        RuntimeError: If Terraform exits with an error (the output is included).
    """
    env = {**os.environ, "TF_IN_AUTOMATION": "1"}
    # Terraform reads TF_VAR_<name> as the sensitive variable of that name. Set it on the child environment dict:
    # os.environ upper-cases names on Windows and Terraform matches the exact lower-case name.
    env["TF_VAR_neon_connection_string"] = neon_dsn
    env["TF_VAR_groq_api_key"] = groq_key
    env["TF_VAR_groq_model"] = GROQ_MODEL
    process = subprocess.Popen(["terraform", *args, "-no-color"] if args[0] != "output" else ["terraform", *args],
                               cwd=TF_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    lines = []
    for line in process.stdout:
        lines.append(line)
        if args[0] != "output":
            logger.info(line.rstrip())
    if process.wait() != 0:
        raise RuntimeError("terraform failed:\n" + "".join(lines)[-3000:])
    return "".join(lines)


show("Terraform files:", sorted(p.name for p in TF_DIR.glob("*.tf")))

## 4.2 `terraform init` and `terraform plan`

`plan` shows what will be created without creating anything. Read it once: it is the list of resources from the table above.

In [ ]:
VARS = ["-var", f"region={REGION}", "-var", f"code_zip_path={ZIP_PATH.resolve().as_posix()}"]
if AWS_PROFILE:
    VARS += ["-var", f"aws_profile={AWS_PROFILE}"]

terraform("init", "-upgrade")
plan_text = terraform("plan", *VARS)

## 4.3 `terraform apply` — creates the resources

Takes a few minutes (Runtimes and the Gateway wait until they are `READY`). Set `DEPLOY = False` to skip it.

In [ ]:
DEPLOY = True   # False = skip creating AWS resources

if DEPLOY:
    terraform("apply", "-auto-approve", *VARS)
else:
    show("DEPLOY is False -> nothing created")

## 4.4 Read the outputs (ARNs and URLs)

In [ ]:
outputs = {k: v["value"] for k, v in json.loads(terraform("output", "-json")).items()}
for key, value in outputs.items():
    if key != "cognito_client_secret":                  # never log secrets
        show(f"{key:24}", value)

# Part 5. Verify every layer on AWS

We test from the outside in, one layer at a time, so a failure tells you exactly which part is wrong.

Helper for all calls below: our own `SigV4Auth` from `aws_common.py` (the same file the supervisor uses).

In [ ]:
sys.path.insert(0, str(APP_DIR.resolve()))
from aws_common import SESSION_HEADER, SigV4Auth, new_session_id    # the file we wrote in 1.1

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

## 5.1 The MCP server on Runtime (calling it directly with a Cognito token)

> 🔶 **AWS CHANGE** — locally we called `http://127.0.0.1:8001/mcp` with no credentials. Now the URL is the **Runtime
> invocation URL** and the server is protected by a **JWT**: we first ask Cognito for a token with the *client-credentials*
> flow (the same thing the Gateway will do for us in 5.2), then send it as `Authorization: Bearer ...`.
> The MCP code is otherwise identical: `initialize` -> `list_tools` -> `call_tool`.

In [ ]:
token_response = httpx.post(
    outputs["cognito_token_endpoint"],
    auth=(outputs["cognito_client_id"], outputs["cognito_client_secret"]),
    data={"grant_type": "client_credentials", "scope": "hm-mcp/invoke"},
    timeout=30,
)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]
show("got a Cognito access token, expires in (s):", token_response.json()["expires_in"])

async with streamablehttp_client(outputs["mcp_url"], headers={"Authorization": f"Bearer {access_token}"}) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        show("tools on the MCP Runtime:", [t.name for t in tools.tools])
        result = await session.call_tool("list_orders", {"customer_id": "cust001"})
        show("list_orders ->", result.content[0].text)

## 5.2 The Gateway (one URL in front of the MCP server)

> 🔶 **AWS CHANGE** — this is the Gateway from Part 6 of the local notebook, now real.
>
> - We call the **Gateway URL** (`.../mcp`), signing with **SigV4** (inbound auth = IAM). No token, no knowledge of the MCP server's address.
> - Behind the scenes the Gateway got a Cognito token through AgentCore Identity and called the MCP Runtime for us.
> - **Look at the tool names**: they are prefixed with the target name (`order-management___list_orders`). That is how a Gateway keeps
>   tools of different targets apart, and why `find_tool` in the supervisor matches on the suffix.

In [ ]:
GATEWAY_MCP_URL = outputs["gateway_url"]          # the Terraform output already ends in /mcp

async with streamablehttp_client(GATEWAY_MCP_URL, auth=SigV4Auth(region=REGION)) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        names = [t.name for t in tools.tools]
        show("tools seen through the Gateway:", names)
        result = await session.call_tool(next(n for n in names if n.endswith("___list_orders")), {"customer_id": "cust001"})
        show("list_orders via Gateway ->", result.content[0].text)

## 5.3 The shipment A2A agent on Runtime

> 🔶 **AWS CHANGE** — locally: `http://127.0.0.1:9000`, no auth. Now: the Runtime URL, **SigV4-signed**, with a **session id header**,
> and the **card URL overridden** (the card cannot know the Runtime ARN, see 1.3). The streamed events (`working` progress lines,
> then the artifact) are the same as before: A2A streaming passes through Runtime unchanged.

In [ ]:
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.helpers.proto_helpers import get_artifact_text, get_message_text, new_text_part
from a2a.types.a2a_pb2 import Message, Role, SendMessageRequest, TaskState


async def ask_shipment_agent(text: str, customer_id: str):
    """Send a message to the deployed A2A agent and yield what it streams back.

    Args:
        text: The customer's message.
        customer_id: Customer id sent as request metadata.

    Yields:
        tuple[str, str]: `("progress" | "answer" | "state", text)`.
    """
    async with httpx.AsyncClient(timeout=90, auth=SigV4Auth(region=REGION), headers={SESSION_HEADER: new_session_id()}) as http:
        card = await A2ACardResolver(http, outputs["shipment_url"]).get_agent_card()
        show("agent card name:", card.name)
        card.supported_interfaces[0].url = outputs["shipment_url"]            # override with the real Runtime URL
        client = ClientFactory(ClientConfig(streaming=card.capabilities.streaming, httpx_client=http)).create(card)
        request = SendMessageRequest(
            message=Message(message_id=f"msg-{time.time_ns()}", role=Role.ROLE_USER, parts=[new_text_part(text)]),
            metadata={"customer_id": customer_id},
        )
        async for response in client.send_message(request):
            if response.HasField("status_update"):
                status = response.status_update.status
                if status.state == TaskState.TASK_STATE_WORKING and status.message.parts:
                    yield "progress", get_message_text(status.message)
                elif status.state == TaskState.TASK_STATE_COMPLETED:
                    yield "state", TaskState.Name(status.state)
            elif response.HasField("artifact_update"):
                yield "answer", get_artifact_text(response.artifact_update.artifact, delimiter="")
        await client.close()


async for kind, value in ask_shipment_agent("Where is my order ORD-1007?", "cust001"):
    show(f"[{kind:8}]", value)

## 5.4 The whole thing: the supervisor on Runtime

> 🔶 **AWS CHANGE** — locally we `POST`ed to `http://127.0.0.1:8090/invocations`. On AWS you call the **`InvokeAgentRuntime` API**
> (boto3 `bedrock-agentcore` client) with the Runtime ARN, a **session id** and the same JSON body. AWS does the SigV4 signing for you;
> the response is the same stream of server-sent events (`data: {...}`) that the code `yield`s.
>
> The first call after a quiet period is slower: Runtime starts a fresh microVM (a *cold start*).

In [ ]:
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)


def invoke_supervisor(message: str, customer_id: str = "cust001", session_id: str | None = None) -> None:
    """Call the deployed supervisor and log each streamed event.

    Args:
        message: The customer's message.
        customer_id: Customer id sent in the body (a real system takes it from the login token instead).
        session_id: Runtime session id (33+ chars); a new conversation per call if omitted.
    """
    response = agentcore.invoke_agent_runtime(
        agentRuntimeArn=outputs["supervisor_runtime_arn"],
        runtimeSessionId=session_id or new_session_id(),
        payload=json.dumps({"message": message, "customer_id": customer_id}).encode(),
        contentType="application/json",
        accept="text/event-stream",
    )
    for line in response["response"].iter_lines():
        text = line.decode() if isinstance(line, bytes) else line
        if text.startswith("data:"):
            event = json.loads(text.removeprefix("data:").strip())
            show(f"  {event.get('type'):15}", event.get("agent", "") + " " + event.get("text", ""))


for question in ["What was my last purchase?", "Where is my order ORD-1007?", "What is the weather like?"]:
    show(f"Q: {question}")
    invoke_supervisor(question)

### What just happened, end to end

```
you --InvokeAgentRuntime--> supervisor Runtime
        order question  : supervisor --SigV4--> Gateway --OAuth--> MCP Runtime --> tool result
        shipping question: supervisor --SigV4--> shipment Runtime (A2A) --> streamed progress + answer
```

Compare with the local notebook: the supervisor's **logic did not change**. What changed is *where things run* and *how they
prove who they are*: IAM (SigV4) between AWS services, a Cognito JWT between the Gateway and the MCP server.

# Part 6. When something fails: read the Runtime logs

> 🔶 **AWS CHANGE** — locally you read `agentcore_demo/*.log`. On AWS every Runtime writes to CloudWatch Logs in the group
> `/aws/bedrock-agentcore/runtimes/<runtime-id>-DEFAULT`. This helper prints the latest lines of any Runtime.

In [ ]:
def tail_runtime_logs(runtime_arn: str, minutes: int = 10, limit: int = 25) -> None:
    """Print the most recent CloudWatch log lines of a Runtime.

    Args:
        runtime_arn: ARN of the Runtime (from the Terraform outputs).
        minutes: How far back to look.
        limit: Maximum number of lines to print.
    """
    runtime_id = runtime_arn.split("/")[-1]
    group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"
    logs = boto3.client("logs", region_name=REGION)
    try:
        events = logs.filter_log_events(logGroupName=group, startTime=int((time.time() - minutes * 60) * 1000), limit=limit)["events"]
    except logs.exceptions.ResourceNotFoundException:
        show("no log group yet:", group)
        return
    show(f"--- {group} ({len(events)} lines)")
    for event in events:
        show("  ", event["message"].rstrip()[:200])


tail_runtime_logs(outputs["supervisor_runtime_arn"])

# Part 7. Destroy everything

Runtimes only bill while active, but the Gateway, Cognito and S3 objects are resources you do not want to leave around.
`terraform destroy` removes every resource listed in Part 4 (the S3 bucket is emptied automatically: `force_destroy = true`).

Set `DESTROY = True` and run the cell when you are done experimenting.

In [ ]:
DESTROY = False   # flip to True to delete the whole stack

if DESTROY:
    terraform("destroy", "-auto-approve", *VARS)
else:
    show("DESTROY is False -> stack left running. Remember to destroy it when you are done.")

## Summary: every local-vs-AWS change in one table

| Area | Local notebook | AWS notebook |
|---|---|---|
| MCP server start | `127.0.0.1:8001`, stateful | `0.0.0.0:8000`, `stateless_http=True`, JWT in front (Runtime) |
| LLM | Groq `openai/gpt-oss-20b` (`GROQ_API_KEY`, `GROQ_MODEL` from `.env` / config) | same code; key + model are sensitive Terraform variables -> supervisor Runtime env |
| MCP data | Neon via `NEON_POSTGRES_CONNECTION_STRING` from `.env` | same queries; the connection string is a sensitive Terraform variable -> Runtime env var |
| Order agent -> tools | direct MCP URL | Gateway URL, SigV4, prefixed tool names (`find_tool`) |
| Gateway -> MCP | n/a | Cognito client-credentials token via AgentCore Identity |
| Shipment A2A | `127.0.0.1:9000` | `0.0.0.0:9000`, `/ping`, IAM auth, URL override of the card, default `A2A-Version` middleware |
| Supervisor | `127.0.0.1:8090`, `POST /invocations` | `0.0.0.0:8080`, `InvokeAgentRuntime`, session id, execution role |
| Packaging | none | arm64 zip in S3 (`uv pip install --python-platform aarch64-manylinux_2_28`) |
| Infra | none | Terraform: S3, IAM x3, Cognito, credential provider, 3 Runtimes, Gateway + target |
| Logs | local files | CloudWatch Logs per Runtime |

## Where to go next

1. Give the shipment agent an LLM too, and add conversation memory per Runtime session.
2. Replace Gateway **IAM** inbound auth with a **JWT** from your login provider (Auth0/Cognito), so end-user identity reaches the agents.
3. Add a second Gateway target (for example a Lambda) and see both tool sets appear in one `list_tools()`.